In [1]:
import glob, pandas as pd, numpy as np

print(glob.glob("/kaggle/input/**/iemocap_wada_snr*.csv", recursive=True))
print(glob.glob("/kaggle/input/**/B2/*.csv", recursive=True))

snr = pd.read_csv(glob.glob("/kaggle/input/**/iemocap_wada_snr.csv", recursive=True)[0])
print(snr.shape, snr.columns.tolist()); print(snr.iloc[0].to_dict())
print(snr.iloc[:, 1].value_counts().head(8))
print(snr.snr_db.describe(), snr.snr_db.isna().sum(), (snr.snr_db >= 99).sum())

b2 = pd.read_csv(glob.glob("/kaggle/input/**/B2/*.csv", recursive=True)[0])
print(b2.shape, b2.columns.tolist()); print(b2.iloc[0].to_dict())

['/kaggle/input/datasets/mdkaifraza123/meld-wada-snr/iemocap_wada_snr.csv']
['/kaggle/input/iemocap-denoised-dataset/B2/processed_iemocap_audio_features_5.csv', '/kaggle/input/iemocap-denoised-dataset/B2/iemocap_audio_features_denoised.csv']
(10190, 4) ['file', 'session', 'path', 'snr_db']
{'file': 'Ses01F_impro01.wav', 'session': 'Session1', 'path': '/kaggle/input/datasets/dejolilandry/iemocapfullrelease/IEMOCAP_full_release/Session1/dialog/wav/Ses01F_impro01.wav', 'snr_db': 19.88667259361197}
session
sentences    10039
Session3        32
Session5        31
Session2        30
Session4        30
Session1        28
Name: count, dtype: int64
count    10190.000000
mean        21.727622
std         13.403748
min        -20.000000
25%         14.885749
50%         19.000000
75%         25.000000
max        100.000000
Name: snr_db, dtype: float64 0 127
(10039, 25) ['video_id', 'segment_id', 'emotion', 'gender', 'mode', 'text', 'duration', 'duration_standardized', 'avg_intensity_standardized'

In [2]:
snr = snr[snr.session == "sentences"].copy()
snr["snr_db"] = snr.snr_db.clip(-10, 40)
print(len(snr), snr.file.nunique(), snr.snr_db.isna().sum())
print(snr[["file", "path"]].head(3).to_dict("records"))

den = pd.read_csv("/kaggle/input/iemocap-denoised-dataset/B2/iemocap_audio_features_denoised.csv")
print(den.shape, den.columns.tolist()); print(den.iloc[0].to_dict())
print(b2["mode"].value_counts())
print((b2[["video_id", "segment_id"]].values == den[["video_id", "segment_id"]].values).all() if len(den) == len(b2) else "different length")

10039 10039 0
[{'file': 'Ses01F_impro01_F000.wav', 'path': '/kaggle/input/datasets/dejolilandry/iemocapfullrelease/IEMOCAP_full_release/Session1/sentences/wav/Ses01F_impro01/Ses01F_impro01_F000.wav'}, {'file': 'Ses01F_impro01_F001.wav', 'path': '/kaggle/input/datasets/dejolilandry/iemocapfullrelease/IEMOCAP_full_release/Session1/sentences/wav/Ses01F_impro01/Ses01F_impro01_F001.wav'}, {'file': 'Ses01F_impro01_F002.wav', 'path': '/kaggle/input/datasets/dejolilandry/iemocapfullrelease/IEMOCAP_full_release/Session1/sentences/wav/Ses01F_impro01/Ses01F_impro01_F002.wav'}]
(10039, 14) ['video_id', 'segment_id', 'emotion', 'gender', 'mode', 'text', 'duration', 'avg_intensity', 'intensity_variation', 'avg_pitch', 'pitch_std', 'pitch_range', 'articulation_rate', 'mean_hnr']
{'video_id': 'Ses01F_script02_1', 'segment_id': 0, 'emotion': 'neu', 'gender': 'F', 'mode': 'train', 'text': 'Fine.', 'duration': 2.07, 'avg_intensity': 54.96897254171316, 'intensity_variation': 8.419417382712814, 'avg_pitch'

In [3]:
import wave
b = b2.copy()
b["n"] = b.sort_values(["video_id", "segment_id"]).groupby(["video_id", "gender"]).cumcount()
b["file"] = b.video_id + "_" + b.gender + b.n.map("{:03d}".format) + ".wav"

m = b.merge(snr[["file", "path", "snr_db"]], on="file", how="left", validate="one_to_one")
assert (m.video_id.values == b2.video_id.values).all() and len(m) == len(b2)
print("no SNR match:", m.snr_db.isna().sum())

def dur(p):
    try:
        with wave.open(p) as w: return w.getnframes() / w.getframerate()
    except Exception: return np.nan
m["wav_dur"] = m.path.map(dur)
d = (m.wav_dur - m.duration).abs()
print("unreadable wavs:", d.isna().sum()); print(d.describe())

no SNR match: 0
unreadable wavs: 10039
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
dtype: float64


In [4]:
import os
tr = m[m["mode"] == "train"].snr_db
lo, hi = tr.quantile([1/3, 2/3]); print("band edges (dB):", lo, hi)
m["snr_band"] = np.where(m.snr_db <= lo, "low", np.where(m.snr_db <= hi, "moderate", "high"))
print(m.groupby("mode").snr_band.value_counts(normalize=True).round(3))

HEDGE = {"high": "", "moderate": "likely ", "low": "possibly "}
def snr_description(desc, snr_db, band):
    body = desc.replace("Target speech characteristics: ", "").rstrip(".")
    parts = [p.strip() for p in body.split(", ")]
    body = ", ".join(HEDGE[band] + p for p in parts)
    return f"Target speech characteristics (estimated SNR {snr_db:.0f} dB, {band} audio reliability): {body}."

src = "/kaggle/input/iemocap-denoised-dataset/B2/processed_iemocap_audio_features_5.csv"
out_dir = "/kaggle/working/B3_iemocap"; os.makedirs(out_dir, exist_ok=True)
out = b2.copy()
out["description"] = [snr_description(a, s, bd) for a, s, bd in zip(b2.description, m.snr_db, m.snr_band)]
out.to_csv(f"{out_dir}/processed_iemocap_audio_features_5.csv", index=False)

ref = pd.read_csv(src, dtype=str, keep_default_na=False)
chk = pd.read_csv(f"{out_dir}/processed_iemocap_audio_features_5.csv", dtype=str, keep_default_na=False)
bad = [c for c in ref.columns if c != "description" and not ref[c].equals(chk[c])]
print("shape same:", ref.shape == chk.shape, "| differing non-description cols:", bad)
for band in ["high", "moderate", "low"]:
    print(chk[chk.description.str.contains(f"{band} audio")].description.iloc[0])

band edges (dB): 16.0 23.0
mode   snr_band
test   moderate    0.368
       low         0.357
       high        0.276
train  moderate    0.342
       low         0.337
       high        0.322
valid  low         0.369
       high        0.336
       moderate    0.295
Name: proportion, dtype: float64
shape same: True | differing non-description cols: ['duration_standardized', 'avg_intensity_standardized', 'intensity_variation_standardized', 'avg_pitch_standardized', 'pitch_std_standardized', 'pitch_range_standardized', 'articulation_rate_standardized', 'mean_hnr_standardized']
Target speech characteristics (estimated SNR 30 dB, high audio reliability): moderate volume with moderate variation, very low pitch with low variation, moderate speaking rate.
Target speech characteristics (estimated SNR 20 dB, moderate audio reliability): likely moderate volume with moderate variation, likely low pitch with moderate variation, likely low speaking rate.
Target speech characteristics (estimated SN